In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import joblib

In [ ]:
data = {

    "opportunity_id": [
        1, 2, 3, 4, 5,
        6, 7, 8, 9, 10
    ],


    "company": [
        "Company A",
        "Company B",
        "Company C",
        "Company D",
        "Company E",
        "Company F",
        "Company G",
        "Company H",
        "Company I",
        "Company J"
    ],


    "required_skills": [

        "python machine learning artificial intelligence",

        "javascript react frontend html css",

        "python sql data analysis",

        "java spring boot backend database",

        "python tensorflow deep learning",

        "cybersecurity linux network security",

        "python data science machine learning",

        "react javascript ui ux",

        "cloud aws docker devops",

        "python artificial intelligence deep learning"
    ]

}


df = pd.DataFrame(data)
df.head()

In [ ]:
vectorizer = TfidfVectorizer()
skill_vectors = vectorizer.fit_transform(
    df["required_skills"]
)


skills = vectorizer.get_feature_names_out()


print("Market Trends AI Model trained successfully")

print(
    "Number of detected skills:",
    len(skills)
)

In [ ]:
from collections import Counter
all_skills = []


for skills_text in df["required_skills"]:

    all_skills.extend(
        skills_text.split()
    )


skill_counts = Counter(
    all_skills
)

sorted_skills = skill_counts.most_common()



market_trends = {

    "high_demand": [],

    "medium_demand": [],

    "low_demand": []
}



total = len(sorted_skills)


for index, (skill, count) in enumerate(sorted_skills):

    if index < total * 0.3:

        market_trends["high_demand"].append(
            skill
        )


    elif index < total * 0.7:

        market_trends["medium_demand"].append(
            skill
        )


    else:

        market_trends["low_demand"].append(
            skill
        )

joblib.dump(
    vectorizer,
    "market_vectorizer.pkl"
)

joblib.dump(
    market_trends,
    "market_trends.pkl"
)

print(
    "Market Trends AI Model saved successfully"
)
market_trends

In [ ]:
from fastapi import FastAPI
import joblib

app = FastAPI(
    title="Market Trends AI API",
    description="Analyze market skill demand using AI",
    version="1.0"
)

vectorizer = joblib.load(
    "market_vectorizer.pkl"
)


market_trends = joblib.load(
    "market_trends.pkl"
)

print("Market Trends AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }

@app.post("/market-trends")
def market_trends_api(data: dict):


    opportunities = data.get(
        "opportunities",
        []
    )


    all_skills = []

    for opportunity in opportunities:

        skills = opportunity.get(
            "requiredSkills",
            []
        )

        all_skills.extend(
            [
                skill.lower()
                for skill in skills
            ]
        )


    result = {

        "high_demand": [],

        "medium_demand": [],

        "low_demand": []
    }

    for skill in all_skills:

        if skill in market_trends["high_demand"]:

            result["high_demand"].append(
                skill
            )


        elif skill in market_trends["medium_demand"]:

            result["medium_demand"].append(
                skill
            )


        else:

            result["low_demand"].append(
                skill
            )

    for key in result:

        result[key] = list(
            set(result[key])
        )
    return {

        "market_skill_trends":
            result
    }

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)


await server.serve()